# Fine-tune `harpreetsahota/car-dd-segmentation-yolov11` on VehiDE

A fresh attempt at this model, incorporating what earlier attempts found:

1. **OOM at imgsz=1280 + multi_scale=True, even at batch=1.** YOLO11x-seg is ~5.3x more parameters / ~7.4x more GFLOPs than the `cardd-yolov8s` model this project fine-tuned successfully. `multi_scale=True` made it worse by pushing image size *above* the configured value on some batches. This run uses a fixed `imgsz=1024`, no multi-scale.
2. **Only 3 of 6 classes transferred from the pretrained checkpoint** (`"Remapped 3/6 cls head rows from pretrained weights by class name"` in an earlier attempt's log) — Ultralytics matches class heads by exact name string, and this checkpoint's own class names likely don't match this project's naming convention word-for-word. This notebook checks that explicitly, prints both name lists side by side, and **stops with a clear warning** if not all 6 match, rather than silently proceeding with half the head randomly initialised again.
3. **`epochs=150` (reference target) is very unlikely to fit Kaggle's GPU quota** at this model's measured cost (~51 min/epoch at a similar image size) — that's ~127 hours, over 4x a week's quota. This notebook probes real per-epoch cost first and prints an honest total-time estimate before you commit, the same discipline that would have caught the OOM before it happened.
4. **`cache=True` risks a RAM crash** (not a GPU one) at this dataset size and image resolution. This notebook checks available system RAM at runtime and falls back to `cache='disk'` if RAM caching looks unsafe, rather than assuming it fits.
5. **P100 is not compatible with the PyTorch build in this Kaggle environment** (`"CUDA error: no kernel image is available for execution on the device"` — P100 is Pascal architecture, compute capability 6.0, and the installed torch/cu128 build has no compiled kernels for it). This is a hardware/build mismatch, not fixable via training config. **Use T4, not P100** — T4 has been directly confirmed to work with this exact software stack. Section 1 now checks this automatically and stops immediately with a clear message if you pick an incompatible accelerator, rather than failing deep inside a training call.

**Prerequisite:** `VehiDE_Seg_Dataset_Prep.ipynb` has already been run and published.

**Setup on Kaggle:** attach the published segmentation dataset as an input, set the accelerator to **GPU T4** (or T4 x2) — not P100.


## 1. Setup

In [1]:
!pip install -q ultralytics huggingface_hub psutil

import json, shutil, time, os, subprocess
from pathlib import Path

import torch

assert torch.cuda.is_available(), "Enable GPU in Settings \u2192 Accelerator before running."
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM: %.1f GB" % (torch.cuda.get_device_properties(0).total_memory / 1e9))

# Compute-capability check -- catches a GPU/PyTorch-build mismatch immediately,
# rather than after building the whole pipeline down to a real training call.
# ("CUDA error: no kernel image is available for execution on the device" means
# the installed torch build has no compiled kernels for this GPU's architecture --
# this happened with P100 (Pascal, compute capability 6.0) against a recent
# torch/cu128 build on Kaggle. Not fixable via training config; the practical
# fix is switching the accelerator to a GPU this build actually supports.)
device_cap = torch.cuda.get_device_capability(0)
sm_str = f"sm_{device_cap[0]}{device_cap[1]}"
supported = torch.cuda.get_arch_list()
print(f"GPU compute capability: {device_cap} ({sm_str})")
print(f"This torch build's compiled kernel architectures: {supported}")

if sm_str not in supported:
    raise RuntimeError(
        f"This GPU's architecture ({sm_str}) has no compiled kernels in the "
        f"installed PyTorch build (supports: {supported}). Training will fail "
        f"immediately at model.to(device) with 'no kernel image is available "
        f"for execution on the device' -- this is a build/hardware mismatch, "
        f"not something fixable via batch size, imgsz, or any training arg.\n"
        f"Fix: change Settings -> Accelerator to a different GPU (T4 has been "
        f"confirmed to work with this exact software stack earlier in this "
        f"project) and re-run from this cell."
    )
print("GPU/PyTorch build compatible.")

import psutil
ram_gb = psutil.virtual_memory().total / 1e9
ram_available_gb = psutil.virtual_memory().available / 1e9
print(f"System RAM: {ram_gb:.1f} GB total, {ram_available_gb:.1f} GB available")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 52.9 MB/s eta 0:00:00
GPU: Tesla T4
VRAM: 15.6 GB
GPU compute capability: (7, 5) (sm_75)
This torch build's compiled kernel architectures: ['sm_70', 'sm_75', 'sm_80', 'sm_86', 'sm_90', 'sm_100', 'sm_120']
GPU/PyTorch build compatible.
System RAM: 33.7 GB total, 31.8 GB available


## 2. Mount the converted segmentation dataset

In [2]:
!ls /kaggle/input/

# EDIT if your dataset is mounted at a different path — confirmed structure:
#   <SEG_ROOT>/vehide_seg/images/{train,val,test}/
#   <SEG_ROOT>/vehide_seg/labels/{train,val,test}/
#   <SEG_ROOT>/damage-seg.yaml          <- sibling of vehide_seg/, NOT nested inside it
SEG_ROOT = Path("/kaggle/input/datasets/m4rcuseryx/vehide-segmentation-dataset")
SEG_IMAGES_LABELS_ROOT = SEG_ROOT / "vehide_seg"
SEG_DATA_YAML_SOURCE = SEG_ROOT / "damage-seg.yaml"

assert SEG_ROOT.exists(), "Segmentation dataset not found at this path — check /kaggle/input/ above and update SEG_ROOT."
assert SEG_IMAGES_LABELS_ROOT.exists(), f"Expected images/labels under {SEG_IMAGES_LABELS_ROOT}, not found."
assert SEG_DATA_YAML_SOURCE.exists(), f"Expected damage-seg.yaml at {SEG_DATA_YAML_SOURCE}, not found."

print(open(SEG_DATA_YAML_SOURCE).read())

CLASS_NAMES = ["dent", "scratch", "crack", "broken_lamp", "shattered_glass", "flat_tyre"]

train_n = len(list((SEG_IMAGES_LABELS_ROOT / "images" / "train").glob("*.jpg")))
val_n = len(list((SEG_IMAGES_LABELS_ROOT / "images" / "val").glob("*.jpg")))
print(f"Train images: {train_n}, Val images: {val_n}")

datasets
names:
- dent
- scratch
- crack
- broken_lamp
- shattered_glass
- flat_tyre
nc: 6
path: /kaggle/working/vehide_seg
test: images/test
train: images/train
val: images/val

Train images: 9545, Val images: 2047


In [3]:
import yaml

with open(SEG_DATA_YAML_SOURCE) as f:
    cfg = yaml.safe_load(f)
cfg["path"] = str(SEG_IMAGES_LABELS_ROOT)

SEG_DATA_YAML = "/kaggle/working/damage-seg.yaml"
with open(SEG_DATA_YAML, "w") as f:
    yaml.safe_dump(cfg, f)
print(open(SEG_DATA_YAML).read())

names:
- dent
- scratch
- crack
- broken_lamp
- shattered_glass
- flat_tyre
nc: 6
path: /kaggle/input/datasets/m4rcuseryx/vehide-segmentation-dataset/vehide_seg
test: images/test
train: images/train
val: images/val



## 3. Download the pretrained checkpoint and verify class-name transfer

This is the step that silently under-transferred last time (3 of 6 classes randomly reinitialised). Checked explicitly here — if it's not a clean 6/6 match, this cell **raises** rather than letting you find out after another expensive training run.

In [4]:
from huggingface_hub import hf_hub_download, list_repo_files
from ultralytics import YOLO

files = list_repo_files("harpreetsahota/car-dd-segmentation-yolov11")
print("Files in repo:", files)

# EDIT this filename to match whatever the listing above actually shows
WEIGHTS_FILENAME = "best.pt"
checkpoint_weights = hf_hub_download(repo_id="harpreetsahota/car-dd-segmentation-yolov11", filename=WEIGHTS_FILENAME)
print("Downloaded:", checkpoint_weights)

probe_model = YOLO(checkpoint_weights)
checkpoint_names = probe_model.names
print("\nCheckpoint's own class names:", checkpoint_names)
print("This project's class names:  ", {i: n for i, n in enumerate(CLASS_NAMES)})

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


Files in repo: ['.gitattributes', 'README.md', 'best.pt']


best.pt:   0%|          | 0.00/125M [00:00<?, ?B/s]

Downloaded: /root/.cache/huggingface/hub/models--harpreetsahota--car-dd-segmentation-yolov11/snapshots/f455e7b592b64958e343f457de3d0ba2c79df884/best.pt

Checkpoint's own class names: {0: 'crack', 1: 'dent', 2: 'glass shatter', 3: 'lamp broken', 4: 'scratch', 5: 'tire flat'}
This project's class names:   {0: 'dent', 1: 'scratch', 2: 'crack', 3: 'broken_lamp', 4: 'shattered_glass', 5: 'flat_tyre'}


In [5]:
# Ultralytics remaps a pretrained head to a new class list by matching NAME
# STRINGS exactly. Any name that doesn't match exactly gets a randomly
# initialised head row instead of a transferred one -- silent unless you
# check for it directly, which is what the first attempt on this model missed.
checkpoint_name_set = {str(v).strip().lower() for v in checkpoint_names.values()}
project_name_set = {n.strip().lower() for n in CLASS_NAMES}
matched = checkpoint_name_set & project_name_set
unmatched_ours = project_name_set - checkpoint_name_set
unmatched_theirs = checkpoint_name_set - project_name_set

print(f"Exact-match classes: {len(matched)}/6 -> {sorted(matched)}")
if unmatched_ours:
    print(f"\nOur classes with NO exact match in the checkpoint (will be randomly")
    print(f"initialised, not transferred, unless fixed below): {sorted(unmatched_ours)}")
    print(f"Checkpoint's corresponding unmatched names (likely the same concept,")
    print(f"different string): {sorted(unmatched_theirs)}")
    print(
        "\nACTION NEEDED: compare the two unmatched lists above. If they're the "
        "same underlying classes with different naming (e.g. 'glass shatter' vs "
        "'shattered_glass'), either rename this project's CLASS_NAMES to match "
        "the checkpoint's exact strings before building the data yaml, or edit "
        "checkpoint_names on the loaded model object directly to match ours "
        "before calling .train(). Do not proceed to Section 5 until this says "
        "6/6 matched, or you've made a deliberate, informed choice to proceed "
        "with partial transfer."
    )
else:
    print("\nAll 6 classes matched exactly -- full pretrained transfer expected.")

Exact-match classes: 3/6 -> ['crack', 'dent', 'scratch']

Our classes with NO exact match in the checkpoint (will be randomly
initialised, not transferred, unless fixed below): ['broken_lamp', 'flat_tyre', 'shattered_glass']
Checkpoint's corresponding unmatched names (likely the same concept,
different string): ['glass shatter', 'lamp broken', 'tire flat']

ACTION NEEDED: compare the two unmatched lists above. If they're the same underlying classes with different naming (e.g. 'glass shatter' vs 'shattered_glass'), either rename this project's CLASS_NAMES to match the checkpoint's exact strings before building the data yaml, or edit checkpoint_names on the loaded model object directly to match ours before calling .train(). Do not proceed to Section 5 until this says 6/6 matched, or you've made a deliberate, informed choice to proceed with partial transfer.


## 4. Configuration

Starting point is your reference hyperparameters, with `multi_scale` deliberately left off (it contributed directly to the previous OOM) and `epochs`/`cache` treated as targets to be confirmed by the probe in Section 5, not committed blindly.

In [6]:
# RAM-aware cache decision: Ultralytics' cache='ram' loads every training
# image into system memory. Rough estimate for this dataset at imgsz=1024,
# uint8: train_n * ~3.1MB/image. Compare against actually available RAM
# (checked in Section 1) rather than assuming "if RAM allows" is satisfied.
est_cache_ram_gb = train_n * (1024 * 1024 * 3) / 1e9
print(f"Estimated RAM cost of cache='ram' for the training split: ~{est_cache_ram_gb:.1f} GB")
print(f"Available RAM: {ram_available_gb:.1f} GB")

if est_cache_ram_gb < ram_available_gb * 0.6:   # leave real headroom, not just "technically fits"
    CACHE_MODE = True    # RAM caching
    print("-> Using cache=True (RAM). Comfortable headroom available.")
else:
    CACHE_MODE = "disk"  # safer fallback: still faster than no caching, no RAM-crash risk
    print("-> RAM caching looks risky at this dataset size vs. available RAM. "
          "Using cache='disk' instead (still faster than no caching, no crash risk).")

Estimated RAM cost of cache='ram' for the training split: ~30.0 GB
Available RAM: 31.8 GB
-> RAM caching looks risky at this dataset size vs. available RAM. Using cache='disk' instead (still faster than no caching, no crash risk).


In [7]:
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ["KAGGLE_USERNAME"] = secrets.get_secret("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = secrets.get_secret("KAGGLE_KEY")

auth_check = subprocess.run(
    ["kaggle", "datasets", "list", "-s", "zzz_auth_check_zzz", "-p", "1"],
    capture_output=True, text=True,
)
if auth_check.returncode != 0:
    print("STDOUT:", auth_check.stdout)
    print("STDERR:", auth_check.stderr)
    raise RuntimeError(
        "Kaggle API authentication failed. Checkpoint backups will NOT work "
        "until this is fixed. Check:\n"
        "  1. Add-ons -> Secrets -> confirm KAGGLE_USERNAME and KAGGLE_KEY are "
        "both added AND toggled ON for this notebook.\n"
        "  2. The values match your actual kaggle.json exactly.\n"
        "  3. Consider upgrading the kaggle package: !pip install -q -U kaggle"
    )
print("Kaggle API authentication OK.")

RUN_NAME = "yolo11x_cardd_seg_p100"

SESSION_EPOCH_BUDGET = 999   # never self-stop; periodic backup is the real safety net
BACKUP_EVERY_N_EPOCHS = 5
BACKUP_SLUG = "yolo11x-cardd-p100-checkpoint-backup"   # distinct from every prior run's backup identity
BACKUP_DATASET_ID = f"{os.environ['KAGGLE_USERNAME']}/{BACKUP_SLUG}"

BACKUP_STAGE = Path("/kaggle/working/checkpoint_backup_yolo11x_p100")
BACKUP_STAGE.mkdir(parents=True, exist_ok=True)

# epochs=150 is the reference target, not a commitment -- Section 5 measures
# real per-epoch cost on P100 and tells you whether 150 is achievable, or
# what to lower it to, before Section 6 spends any real GPU time on it.
TRAIN_ARGS = dict(
    data=SEG_DATA_YAML,
    epochs=150,
    imgsz=1024,
    batch=4,
    optimizer="AdamW",
    lr0=0.0001,
    lrf=0.01,
    cos_lr=True,
    weight_decay=0.0005,
    warmup_epochs=10,
    cls=0.3,
    dfl=1.7,
    dropout=0.1,
    multi_scale=False,     # deliberately off -- contributed directly to the prior OOM
    patience=20,
    save_period=10,
    amp=True,
    cache=CACHE_MODE,      # decided above from actual available RAM, not assumed
    workers=4,
    close_mosaic=10,
    seed=42,
    deterministic=True,
    plots=True,
    project="/kaggle/working/runs/yolo11x_cardd_seg_p100",
)
print(json.dumps(TRAIN_ARGS, indent=2))

Kaggle API authentication OK.
{
  "data": "/kaggle/working/damage-seg.yaml",
  "epochs": 150,
  "imgsz": 1024,
  "batch": 4,
  "optimizer": "AdamW",
  "lr0": 0.0001,
  "lrf": 0.01,
  "cos_lr": true,
  "weight_decay": 0.0005,
  "warmup_epochs": 10,
  "cls": 0.3,
  "dfl": 1.7,
  "dropout": 0.1,
  "multi_scale": false,
  "patience": 20,
  "save_period": 10,
  "amp": true,
  "cache": "disk",
  "workers": 4,
  "close_mosaic": 10,
  "seed": 42,
  "deterministic": true,
  "plots": true,
  "project": "/kaggle/working/runs/yolo11x_cardd_seg_p100"
}


## 5. Probe first — real per-epoch time AND peak VRAM, on this exact config

Unlike the first attempt's probe (which used its own hardcoded settings, not `TRAIN_ARGS`, and only measured time), this one runs the *exact* `TRAIN_ARGS` on a small subsample and reports both timing and peak VRAM — the second is what would have caught the OOM before it happened.

In [8]:
import random

PROBE_DIR = Path("/kaggle/working/probe_subsample")
probe_img_dir = PROBE_DIR / "images" / "train"
probe_lbl_dir = PROBE_DIR / "labels" / "train"
probe_img_dir.mkdir(parents=True, exist_ok=True)
probe_lbl_dir.mkdir(parents=True, exist_ok=True)

all_train_imgs = sorted((SEG_IMAGES_LABELS_ROOT / "images" / "train").glob("*.jpg"))
sample = random.Random(42).sample(all_train_imgs, min(300, len(all_train_imgs)))
for img_path in sample:
    (probe_img_dir / img_path.name).symlink_to(img_path)
    lbl_path = SEG_IMAGES_LABELS_ROOT / "labels" / "train" / f"{img_path.stem}.txt"
    if lbl_path.exists():
        (probe_lbl_dir / lbl_path.name).symlink_to(lbl_path)

probe_cfg = dict(cfg)
probe_cfg["path"] = str(PROBE_DIR)
probe_cfg["train"] = "images/train"
probe_cfg["val"] = str(SEG_IMAGES_LABELS_ROOT / "images" / "val")

PROBE_YAML = "/kaggle/working/damage-seg-probe.yaml"
with open(PROBE_YAML, "w") as f:
    yaml.safe_dump(probe_cfg, f)
print(f"Probe subsample: {len(sample)} images")

Probe subsample: 300 images


In [9]:
# Mirror TRAIN_ARGS exactly (imgsz, batch, amp, multi_scale, cache) except
# data (the subsample) and epochs (just 1) -- a probe that doesn't match
# production settings can't reliably catch a production OOM.
probe_args = dict(TRAIN_ARGS)
probe_args["data"] = PROBE_YAML
probe_args["epochs"] = 1
probe_args["project"] = "/kaggle/working/runs/probe_yolo11x_p100"
probe_args["name"] = "timing_probe"
probe_args["cache"] = False   # irrelevant for a 1-epoch, 300-image probe; skip the RAM cost here

torch.cuda.reset_peak_memory_stats()
t0 = time.time()
probe_run_model = YOLO(checkpoint_weights)
probe_run_model.train(**probe_args)
probe_wall = time.time() - t0
peak_vram_gb = torch.cuda.max_memory_allocated() / 1e9
total_vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9

print(f"\nMeasured: {probe_wall/60:.1f} min for 1 epoch on {len(sample)} images.")
print(f"Peak VRAM used: {peak_vram_gb:.2f} GB / {total_vram_gb:.1f} GB available "
      f"({100*peak_vram_gb/total_vram_gb:.0f}%)")

if peak_vram_gb > total_vram_gb * 0.85:
    print("\nWARNING: peak VRAM usage is close to the card's limit even on this "
          "small probe. The full training run (larger batches of real-sized "
          "images throughout, not just this subsample) is likely to OOM. "
          "Lower TRAIN_ARGS['imgsz'] or ['batch'] in Section 4 and re-run this "
          "probe before continuing to Section 6.")
else:
    print("\nVRAM headroom looks OK for this config.")

full_epoch_est_min = (probe_wall / 60) * (train_n / len(sample))
print(f"\nEstimated time per epoch on the full {train_n}-image training set: "
      f"~{full_epoch_est_min:.1f} min")
print(f"Estimated total time for TRAIN_ARGS['epochs']={TRAIN_ARGS['epochs']}: "
      f"~{full_epoch_est_min * TRAIN_ARGS['epochs'] / 60:.1f} hours")
print("\nIf that total is impractical against your remaining Kaggle GPU quota, "
      "lower TRAIN_ARGS['epochs'] in Section 4 now and re-run that cell before "
      "continuing to Section 6.")

Ultralytics 8.4.110 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.3, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/damage-seg-probe.yaml, degrees=0.0, deterministic=True, device=, dfl=1.7, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.1, dynamic=False, embed=None, end2end=None, epochs=1, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/root/.cache/huggingface/hub/models--harpreetsahota--car-dd-segmentation-yolov11/snaps

## 6. Fine-tune, with multi-session checkpoint relay

In [ ]:
meta_check = subprocess.run(
    ["kaggle", "datasets", "metadata", BACKUP_DATASET_ID, "-p", "/tmp/meta_check_yolo11x_p100"],
    capture_output=True, text=True,
)
check = meta_check.returncode
resume_from = None

if check == 0:
    print(f"Found existing backup: {BACKUP_DATASET_ID} — downloading...")
    os.makedirs("/kaggle/working/recovered_yolo11x_p100", exist_ok=True)
    dl = subprocess.run(
        ["kaggle", "datasets", "download", BACKUP_DATASET_ID,
         "-p", "/kaggle/working/recovered_yolo11x_p100", "--unzip", "-q"],
        capture_output=True, text=True,
    )
    if dl.returncode != 0:
        print(dl.stdout, dl.stderr)
        raise RuntimeError("Backup dataset metadata was found, but downloading it failed.")
    recovered_pt = list(Path("/kaggle/working/recovered_yolo11x_p100").rglob("last.pt"))
    if recovered_pt:
        resume_from = recovered_pt[0]
        epoch_marker = Path("/kaggle/working/recovered_yolo11x_p100/epoch.txt")
        print(f"Will resume from epoch {epoch_marker.read_text().strip() if epoch_marker.exists() else '?'}")
else:
    print(f"No existing backup dataset ({BACKUP_DATASET_ID}) — session 1, starting fresh "
          f"from {checkpoint_weights}")
    print(f"(kaggle CLI returned: {meta_check.stderr.strip()[:200]})")

In [ ]:
_session_start_epoch = {"value": None}
_dataset_exists = {"value": check == 0}
_backup_failures = []


def _push_backup(trainer, completed_epoch):
    last_pt = trainer.save_dir / "weights" / "last.pt"
    if not last_pt.exists():
        return
    shutil.copy(last_pt, BACKUP_STAGE / "last.pt")
    for extra in ("results.csv", "args.yaml"):
        p = trainer.save_dir / extra
        if p.exists():
            shutil.copy(p, BACKUP_STAGE / extra)
    (BACKUP_STAGE / "epoch.txt").write_text(str(completed_epoch))
    (BACKUP_STAGE / "dataset-metadata.json").write_text(json.dumps({
        "title": "YOLO11x CarDD-seg P100 fine-tune checkpoint backup",
        "id": BACKUP_DATASET_ID,
        "licenses": [{"name": "CC0-1.0"}],
    }))

    if not _dataset_exists["value"]:
        result = subprocess.run(
            ["kaggle", "datasets", "create", "-p", str(BACKUP_STAGE), "--dir-mode", "zip", "-q"],
            capture_output=True, text=True,
        )
        if result.returncode == 0:
            _dataset_exists["value"] = True
    else:
        result = subprocess.run(
            ["kaggle", "datasets", "version", "-p", str(BACKUP_STAGE),
             "-m", f"epoch {completed_epoch}", "--dir-mode", "zip", "-q"],
            capture_output=True, text=True,
        )

    if result.returncode == 0:
        print(f"Backed up checkpoint at epoch {completed_epoch} -> {BACKUP_DATASET_ID}")
    else:
        msg = f"BACKUP FAILED at epoch {completed_epoch}: {result.stderr.strip()[:300]}"
        print(f"\n{'='*70}\n{msg}\n{'='*70}\n")
        _backup_failures.append((completed_epoch, result.stderr.strip()))


def relay_callback(trainer):
    completed = trainer.epoch + 1
    if _session_start_epoch["value"] is None:
        _session_start_epoch["value"] = completed - 1
    epochs_this_session = completed - _session_start_epoch["value"]
    is_budget_stop = epochs_this_session >= SESSION_EPOCH_BUDGET
    is_periodic_backup = completed % BACKUP_EVERY_N_EPOCHS == 0

    if is_budget_stop or is_periodic_backup:
        _push_backup(trainer, completed)
    if is_budget_stop:
        print(f"\nSession budget reached (total completed: {completed}/{trainer.epochs}). Stopping gracefully.")
        trainer.stop = True


t0 = time.time()
if resume_from is not None:
    model = YOLO(str(resume_from))
    model.add_callback("on_train_epoch_end", relay_callback)
    results = model.train(resume=True)
else:
    model = YOLO(checkpoint_weights)
    model.add_callback("on_train_epoch_end", relay_callback)
    results = model.train(name=RUN_NAME, **TRAIN_ARGS)
wall = time.time() - t0
print(f"\nThis session: {wall/60:.1f} min")

if _backup_failures:
    print(f"\nWARNING: {len(_backup_failures)} backup attempt(s) failed. "
          f"Failed epochs: {[e for e, _ in _backup_failures]}")
else:
    print("\nAll backup attempts this session succeeded.")

## 7. Evaluate (same per-class format as the other runs, for direct comparison)

In [ ]:
m = model.val(data=SEG_DATA_YAML, split="test", imgsz=TRAIN_ARGS["imgsz"])
print("Box    mAP50:", float(m.box.map50), " mAP50-95:", float(m.box.map))
print("Mask   mAP50:", float(m.seg.map50), " mAP50-95:", float(m.seg.map))

import pandas as pd
rows = []
for idx, ci in enumerate(m.box.ap_class_index):
    rows.append({
        "class": CLASS_NAMES[int(ci)],
        "box_mAP50": float(m.box.ap50[idx]),
        "mask_mAP50": float(m.seg.ap50[idx]),
    })
pd.DataFrame(rows).sort_values("mask_mAP50", ascending=False)